# Train the Yosemite CycleGAN and export weights for the demo

Run this on a **GPU runtime** (Runtime → Change runtime type → T4). It downloads the
dataset, trains, exports both generators to ONNX, and gives you the files to commit
into `docs/models/` so the [browser demo](https://akshayaa-403.github.io/yosemite-image-translation-gan/)
starts working.

**What you need:** a Kaggle API token. Kaggle → your profile → Settings → API →
*Create New Token*, which downloads `kaggle.json`. You upload it in step 2.

**How long:** ~3 min/epoch on a T4 at 128px. The defaults below run 60 epochs
(~3 hours), which is enough for a clearly recognisable seasonal change. Colab free
tier disconnects after a while, so checkpoints are written every 5 epochs and
`RESUME` at the bottom picks up where it stopped.

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU - switch the runtime type before continuing'

!git clone --depth 1 https://github.com/akshayaa-403/yosemite-image-translation-gan.git repo
%cd repo
!pip install -q pyyaml scikit-image onnx onnxruntime kaggle

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 2. Kaggle credentials

Run the cell, then pick the `kaggle.json` you downloaded. It stays in this runtime
and is deleted when the session ends.

In [ ]:
import json, pathlib
from google.colab import files

uploaded = files.upload()  # choose kaggle.json
assert 'kaggle.json' in uploaded, 'Upload the file named kaggle.json'

kaggle_dir = pathlib.Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
(kaggle_dir / 'kaggle.json').write_bytes(uploaded['kaggle.json'])
(kaggle_dir / 'kaggle.json').chmod(0o600)  # the kaggle CLI refuses world-readable tokens
print('token installed for user:', json.loads(uploaded['kaggle.json'])['username'])

## 3. Dataset

~200MB. The script normalises whatever layout the download has into
`data/summer2winter_yosemite/{trainA,trainB,testA,testB}` and prints the per-split
counts, so a truncated download is obvious immediately.

In [ ]:
!python scripts/prepare_data.py --source kaggle --move

## 4. Train

Two knobs worth thinking about before you start:

* `EPOCHS` — 60 gives a recognisable result; the paper's 200 is better but ~10 hours.
* `G_CHANNELS` — 64 is the real model (~31MB per direction in ONNX float32). **32 gives
  a ~8MB model**, which is a much kinder download for the demo page and still looks
  decent at 128px. Pick 32 unless you specifically want the full-size generator.

Sample grids land in `runs/colab/samples/` — look at them while it trains rather than
waiting for the end.

In [ ]:
EPOCHS = 60
G_CHANNELS = 32   # 32 -> ~8MB ONNX per direction; 64 -> ~31MB
BATCH_SIZE = 4    # a T4 handles 4 at 128px; drop to 2 if you hit OOM

!python scripts/train.py \
  --config configs/yosemite_128.yaml \
  --epochs {EPOCHS} \
  --g-base-channels {G_CHANNELS} \
  --batch-size {BATCH_SIZE} \
  --output-dir runs/colab

In [ ]:
# Look at the newest sample grid: real | translated | back again, both directions.
import glob
from IPython.display import Image, display

grids = sorted(glob.glob('runs/colab/samples/*.png'))
print(f'{len(grids)} sample grids')
if grids:
    display(Image(grids[-1]))

### If the runtime disconnected

Re-run cells 1–3, then this instead of cell 4. It restarts from the last checkpoint.

In [ ]:
# !python scripts/train.py --config configs/yosemite_128.yaml --epochs 60 \
#   --g-base-channels 32 --output-dir runs/colab \
#   --resume runs/colab/checkpoints/latest.pt

## 5. Check it numerically

Cycle-reconstruction metrics plus FID against the held-out split. FID here is a
*relative* signal between your own checkpoints — the test split has a few hundred
images against the 50k FID assumes.

In [ ]:
!pip install -q torchmetrics
!python scripts/evaluate.py --checkpoint runs/colab/checkpoints/latest.pt --fid

## 6. Export for the browser

Writes `docs/models/{summer2winter,winter2summer}.onnx` plus the `manifest.json` the
page reads, and verifies each graph against PyTorch before saving. Add `--quantize`
for int8 (~4x smaller, slight quality loss) if the total is still large.

In [ ]:
!python scripts/export_onnx.py --checkpoint runs/colab/checkpoints/latest.pt

# Sample photos for the demo's gallery + the README strips.
!python scripts/make_demo_assets.py --checkpoint runs/colab/checkpoints/latest.pt --count 6

## 7. Download and commit

This packs everything the demo needs into one zip. Unpack it at the root of your
local clone (it contains `docs/models/` and `docs/samples/`), commit, and push —
the Pages workflow redeploys and the demo goes live.

In [ ]:
import shutil, os
from google.colab import files

shutil.make_archive('/content/demo_assets', 'zip', root_dir='.', base_dir='docs')
size_mb = os.path.getsize('/content/demo_assets.zip') / 1e6
print(f'demo_assets.zip: {size_mb:.1f} MB')
if size_mb > 90:
    print('WARNING: over ~90MB. Re-run the export with --quantize, or train with '
          '--g-base-channels 32, before committing: GitHub rejects files above 100MB.')
files.download('/content/demo_assets.zip')

# Keep the checkpoint too if you might want to resume or re-export later (~90MB).
# files.download('runs/colab/checkpoints/latest.pt')

```bash
# locally, at the repo root
unzip -o ~/Downloads/demo_assets.zip
git add docs/models docs/samples
git commit -m "Publish trained ONNX weights and demo samples"
git push
```

Preview before pushing with `python -m http.server -d docs 8000`.